# 04 — Model Comparison and Out-of-Sample Validation

A 2026 Frontiers systematic review found only **14%** of LMIC AI health models
were externally validated; most were tested on the data they were fitted on.
This notebook is the answer to that (shortcoming #13).

Three questions, in increasing order of how much they matter:

1. Which **backend** and which **ensemble** fit best?
2. How does the model perform **out of sample**, under walk-forward validation
   with a purge gap?
3. Does it beat the **three naive baselines**? If not, critical rule #10 says it
   does not get deployed — regardless of how good its R² looks.

In [ ]:
# Make the repo importable regardless of where Jupyter was launched from.
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src").is_dir() and (p / "config").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import logging
logging.getLogger("afya").setLevel(logging.WARNING)   # keep notebook output readable

import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
print(f"repo root: {ROOT}")

In [ ]:
# Plotting is optional throughout these notebooks: matplotlib is not a hard
# dependency of AFYA-PREDICT, because the platform must install on low-spec
# district hardware. Every notebook falls back to printed tables without it.
#
# Backend selection matters more than it looks. Inside a Jupyter kernel,
# matplotlib configures its own inline backend and we leave it alone. Anywhere
# else - `nbconvert --execute`, CI, a headless server - a GUI backend will block
# forever on a window that never opens (a set-but-unreachable $DISPLAY is enough
# to trigger it), so we force the non-interactive Agg backend.
import os
import sys

try:
    import matplotlib
    _in_kernel = "ipykernel" in sys.modules
    if not os.environ.get("MPLBACKEND") and not _in_kernel:
        matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams["figure.figsize"] = (11, 4)
    plt.rcParams["axes.grid"] = True
    plt.rcParams["grid.alpha"] = 0.3
    HAS_PLT = True
    print(f"matplotlib {matplotlib.__version__} on the "
          f"{matplotlib.get_backend()} backend")
except ImportError:
    HAS_PLT = False
    print("matplotlib not installed - tables will be printed instead of plotted")

In [ ]:
from src.core.config_loader import load_region_config, load_disease_config
from src.core.geo import subset_region

FULL_REGION = load_region_config("tanzania")
print(f"{len(FULL_REGION.districts)} councils across "
      f"{len({d.region for d in FULL_REGION.districts})} regions")

# A small, ecologically diverse subset keeps these notebooks fast to run.
# Swap in FULL_REGION for a national analysis (much slower).
STUDY_DISTRICTS = [
    "Kinondoni",     # dense coastal city
    "Ilala",         # dense coastal city, adjacent to Kinondoni
    "Mwanza City",   # lakeside city
    "Sengerema",     # rural lakeside, low WASH coverage
    "Dodoma City",   # semi-arid central
    "Songea MC",     # southern highlands
]
REGION = subset_region(FULL_REGION, STUDY_DISTRICTS)
pd.DataFrame([d.model_dump() for d in REGION.districts]).set_index("name")

## 1. Set up

In [ ]:
from src.data_ingestion.normalizer import ingest
from src.models.registry import build_module

DISEASE = "malaria"
module = build_module(DISEASE, region=REGION)
SOURCES = sorted(set(module.config.required_sources) | {"dhis2"})

panel = ingest(SOURCES, "2019-W01", "2024-W52", region=REGION)
matrix = module.build_feature_matrix(panel)
usable = matrix.dropna_rows()
print(f"{len(usable.X)} usable district-weeks x {len(usable.feature_names)} features")
print(f"horizon: {module.horizon} weeks")

## 2. Backend comparison

Fitted on a chronological split — never a random one. Shuffling a time series
leaks the future into the validation set and inflates every score.

In [ ]:
from src.models.backends import available_backends, build_regressor
from src.evaluation.metrics import regression_metrics
import time

weeks = usable.weeks
split_week = weeks[int(len(weeks) * 0.75)]
week_index = np.asarray(usable.X.index.get_level_values("week"))
train_mask, test_mask = week_index < split_week, week_index >= split_week

X_train = usable.X[train_mask].to_numpy(dtype=float)
y_train = usable.y[train_mask].to_numpy(dtype=float)
X_test = usable.X[test_mask].to_numpy(dtype=float)
y_test = usable.y[test_mask].to_numpy(dtype=float)
print(f"train {train_mask.sum()} rows (< {split_week}) | test {test_mask.sum()} rows\n")

rows = []
for name in available_backends():
    estimator, info = build_regressor(name, random_state=42)
    started = time.perf_counter()
    try:
        estimator.fit(X_train, y_train)
    except Exception as exc:
        print(f"  {name}: failed to fit ({exc})")
        continue
    fit_seconds = time.perf_counter() - started
    metrics = regression_metrics(y_test, estimator.predict(X_test))
    rows.append({"backend": name, "library": info.library, "fit_seconds": round(fit_seconds, 2),
                 **{k: round(v, 3) for k, v in metrics.items() if k in ("mae", "rmse", "r2", "bias")}})

comparison = pd.DataFrame(rows).sort_values("mae").reset_index(drop=True)
display(comparison)
print("\nMAE is the operational metric: it is in cases, and case counts are what")
print("procurement decisions are sized against.")

## 3. Ensemble

Fusing *models* protects against a model class failing, the same way fusing data
sources protects against a feed failing. Weights are earned on a held-out tail,
not on the training fit.

In [ ]:
from src.models.ensemble import WeightedEnsemble

members = [m for m in module.config.model.ensemble_members] or ["numpy_gbm", "ridge"]
print(f"config asks for: {members}\n")

ensemble = WeightedEnsemble(members).fit(usable.X[train_mask], usable.y[train_mask])
display(pd.DataFrame(ensemble.describe()))

ensemble_metrics = regression_metrics(y_test, ensemble.predict(usable.X[test_mask]))
best_single = comparison.iloc[0]
print(f"\nbest single backend : {best_single['backend']:12} MAE {best_single['mae']:.3f}")
print(f"weighted ensemble   : {'':12} MAE {ensemble_metrics['mae']:.3f}")
verdict = "helps" if ensemble_metrics["mae"] < best_single["mae"] else "does not help here"
print(f"-> the ensemble {verdict}")

### Ensemble disagreement is a real uncertainty signal

Where the members disagree, the forecast is genuinely less certain. That spread
is folded into the interval, so the alert reflects *model* risk as well as data noise.

In [ ]:
spread = ensemble.prediction_spread(usable.X[test_mask])
per_member = ensemble.predict_members(usable.X[test_mask])
print(f"member disagreement: mean {spread.mean():.2f}, "
      f"p90 {np.percentile(spread, 90):.2f} cases")
display(per_member.describe().T.round(2))

## 4. Walk-forward validation

Train on everything up to week *t*, predict *t+1..t+h*, roll forward, repeat.
A **purge gap** equal to the forecast horizon sits between train and test, so a
target built by shifting cases forward cannot appear in both folds.

Every number below is genuinely out of sample.

In [ ]:
from src.evaluation.walk_forward_cv import WalkForwardCV

cv = WalkForwardCV(module, initial_train_weeks=156, step_weeks=26,
                   test_weeks=26, max_folds=4)

splits = list(cv.splits(matrix.weeks, horizon=module.horizon))
print(f"{len(splits)} fold(s):\n")
for i, (train, test) in enumerate(splits):
    gap = matrix.weeks.index(test[0]) - matrix.weeks.index(train[-1])
    print(f"  fold {i}: train {train[0]}..{train[-1]} ({len(train)}w) "
          f"-> gap {gap}w -> test {test[0]}..{test[-1]} ({len(test)}w)")
print(f"\nPurge gap ({gap}w) exceeds the horizon ({module.horizon}w): no target overlap.")

In [ ]:
result = cv.run(matrix)
print(result.summary_line())

print("\nPer-fold accuracy:")
display(pd.DataFrame([{
    "fold": f.fold, "train_end": f.train_weeks[1], "test": f"{f.test_weeks[0]}..{f.test_weeks[1]}",
    "n_test": f.n_test, "mae": round(f.metrics["mae"], 2), "rmse": round(f.metrics["rmse"], 2),
    "r2": round(f.metrics["r2"], 3), "bias": round(f.metrics["bias"], 2),
} for f in result.folds]))

### Stability across folds matters as much as the average

A model with a good mean MAE but wildly varying fold performance is not
trustworthy — it means accuracy depends on which season you happened to test in.

In [ ]:
fold_mae = pd.Series({f.fold: f.metrics["mae"] for f in result.folds})
print(f"MAE across folds: mean {fold_mae.mean():.2f}, "
      f"std {fold_mae.std():.2f}, range {fold_mae.min():.2f}-{fold_mae.max():.2f}")
print(f"coefficient of variation: {fold_mae.std() / fold_mae.mean():.1%}")

if HAS_PLT and not result.predictions.empty:
    sample_district = result.predictions.index.get_level_values("district")[0]
    local = result.predictions.xs(sample_district, level="district").sort_index()
    fig, ax = plt.subplots(figsize=(12, 4))
    x = range(len(local))
    ax.plot(x, local["actual"], label="actual", lw=1.6, color="k")
    ax.plot(x, local["predicted"], label="predicted", lw=1.4)
    ax.fill_between(x, local["lower"], local["upper"], alpha=0.2, label="95% interval")
    ax.set_title(f"Out-of-sample forecast, {sample_district}")
    ax.set_xlabel("out-of-sample week"); ax.set_ylabel("cases"); ax.legend()
    plt.show()

## 5. The baseline gate (critical rule #10)

Three baselines, all of which must be beaten:

| Baseline | What it assumes |
|---|---|
| seasonal naive | this week looks like the same week last year |
| 4-week rolling mean | this week looks like the recent past |
| smoothed persistence | an exponentially weighted AR-style forecast |

A model that cannot beat "same week last year" has learned nothing worth
deploying, however good its R² looks.

In [ ]:
baselines = result.baselines
print(baselines.verdict())
print()
display(baselines.to_frame().round(3))
print(f"\nSkill score (1 - model/baseline error; positive = model wins):")
for name, skill in baselines.skill.items():
    marker = "PASS" if baselines.beaten.get(name) else "FAIL"
    print(f"  {name:20} {skill:+7.1%}   [{marker}]")

## 6. Outbreak detection

Regression error is not the question agencies ask. They ask: when the system said
"outbreak coming", did one arrive, and how much warning did we get?

In [ ]:
outbreak = result.outbreak
display(pd.Series(outbreak.summary()).to_frame("value"))
print(f"\n{outbreak.verdict}")

if outbreak.evaluable:
    sensitivity = outbreak.metrics.get("recall", float("nan"))
    if sensitivity < 0.5 and outbreak.auc >= 0.75:
        print(f"\nNote the gap: AUC {outbreak.auc:.3f} means the model *ranks* outbreak weeks")
        print(f"well, but sensitivity at the configured threshold is only {sensitivity:.0%}.")
        print("A squared-error model regresses towards the mean, so it under-shoots peaks.")
        print(f"The F1-optimal trigger is {outbreak.optimal_threshold:.3f} per 1,000")
        print(f"(configured: {outbreak.threshold_per_1000}). The trend-boost rule in")
        print("config/alert_rules/default.yaml exists for exactly this reason.")

### Choosing an operating point

An agency should pick its own place on the sensitivity/false-alarm trade-off.

In [ ]:
from src.evaluation.outbreak_detection import threshold_sweep

thresholds = np.round(np.linspace(0.4, 2.5, 12) * module.config.alerts.medium, 3)
sweep = threshold_sweep(result.predictions["actual_incidence"],
                        result.predictions["predicted_incidence"], thresholds)
display(sweep[["threshold_per_1000", "sensitivity", "specificity", "precision",
               "f1", "false_alarm_rate"]].round(3))

if HAS_PLT:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(sweep["threshold_per_1000"], sweep["sensitivity"], marker="o", label="sensitivity")
    ax.plot(sweep["threshold_per_1000"], sweep["precision"], marker="s", label="precision")
    ax.plot(sweep["threshold_per_1000"], sweep["f1"], marker="^", label="F1")
    ax.axvline(module.config.alerts.medium, ls="--", c="r", label="configured threshold")
    ax.set_xlabel("alert threshold (cases per 1,000/week)"); ax.legend()
    ax.set_title("Operating-point trade-off")
    plt.show()

## 7. Timeliness — the number that justifies the whole approach

A surveillance system tells you an outbreak started. A forecasting system is only
worth building if it tells you **earlier**.

In [ ]:
display(pd.Series(result.timeliness).to_frame("value"))
lead = result.timeliness.get("mean_lead_time_weeks")
if lead and np.isfinite(lead):
    print(f"\nMean lead time: {lead:.1f} weeks of warning before onset.")
    print("Compare against the response latency measured in notebook 07 - if the")
    print("response takes longer than the warning, the bottleneck is operational,")
    print("not predictive.")
else:
    print("\nNo outbreak onsets in this evaluation window, so lead time is not measurable here.")

## 8. Are the intervals honest?

An earlier version of this platform reported 95% intervals that covered 37% of
outcomes, because the residual quantiles were estimated **in sample**. Walk-forward
validation is what caught it. This section is the regression test.

In [ ]:
from src.evaluation.calibration import interval_calibration, recalibration_factor

calibration = interval_calibration(result.predictions["actual"],
                                   result.predictions["lower"],
                                   result.predictions["upper"])
display(pd.Series(calibration.summary()).to_frame("value"))

if not calibration.is_calibrated:
    factor = recalibration_factor(result.predictions["actual"],
                                  result.predictions["lower"], result.predictions["upper"])
    print(f"\nIntervals need rescaling by {factor:.2f}x - the retraining loop applies this.")
else:
    print("\nIntervals are calibrated within the 10-percentage-point working tolerance.")

print("\nCoverage by forecast magnitude (coverage often degrades at the high end,")
print("which is exactly where the decisions get made):")
display(calibration.reliability.round(3))

## 9. Per-district performance

A national average can hide districts where the model is worse than useless.

In [ ]:
from src.evaluation.benchmark import benchmark_by_district

per_district = benchmark_by_district(result.predictions, horizon=module.horizon)
display(per_district.round(3))

failing = per_district[~per_district["passes"]]
if len(failing):
    print(f"\n{len(failing)} district(s) fail the baseline gate: "
          f"{', '.join(failing['district'])}.")
    print("In production these would fall back to the pooled model or to the")
    print("seasonal baseline, rather than shipping a forecast known to be worse.")
else:
    print("\nEvery district beats all three naive baselines.")

## 10. The acceptance verdict

In [ ]:
import json

report = result.report()
print(json.dumps({k: v for k, v in report.items()
                  if k in ("disease", "horizon_weeks", "folds", "n_predictions",
                           "passes_acceptance")}, indent=2))
print("\nNotes:")
for note in report["acceptance_notes"]:
    print(f"  - {note}")

## Takeaways

* Walk-forward with a purge gap is the only honest evaluation for a forecaster,
  and it is the default here rather than an optional extra.
* The baseline gate is a hard requirement: three naive competitors, all of which
  must lose, per district as well as nationally.
* Reporting AUC alongside sensitivity exposed a real property of the model —
  strong ranking, conservative magnitude — that a single headline metric hides.
* Interval calibration is measured every run, because it is the number most
  likely to be silently wrong.

Next: **`05_spatial_validation.ipynb`** — did the disease spread *where* we said?